# Sensor placement optimization + Isaac Lab (Colab)

This notebook is the **practical** Colab entrypoint for this repository when you use the unofficial Isaac Sim / Isaac Lab Colab installer workflow (same idea as `isaac_lab_2_1_0_colab.ipynb` in this folder).

**What you get end-to-end**
- Installs Isaac Sim + Isaac Lab (same script pattern as the uploaded Colab)
- Clones *this* repo to `/content/sensor_placement_opt`
- Launches a small **HTTP JSON bridge**: `scripts/isaaclab_sensor_bridge.py` (`/health`, `/reconfigure_sensors`, `/run_rollouts`)
- Runs CMA-ES in `sensor_opt` via `IsaacSimEvaluator` over `http://127.0.0.1:<port>`

**Bridge modes (`--bridge-mode`)**
- **`ground`** — `blind_spot_fraction` from depth / point-like observations (with fallbacks) + best-effort collision / success from `info`.
- **`obstacle`** — static obstacle corridor: spawns 3–5 boxes per reset, reports **`t_det_s` / `t_det_s_p95`**, contact-based **collision rate**, and **safety** fields. Use with `configs/obstacle_isaaclab.yaml` (`loss.mode: obstacle_latency`).

**What you should still set for your robot / task**
- In `configs/*.yaml`, set `sensor_models.<lidar|camera>.isaac.prim_path` so `_apply_sensor_config` can move sensor prims (use `{env_idx}` for cloned envs if needed).
- Set `ISAAC_TASK` to an environment ID your Isaac Lab build actually registers.


## Configure your 3D LiDAR + RGB-D mounts in the USD stage (recommended)

`SensorConfig` already describes each sensor (type, mounting slot, offsets, FoV fractions). To actually *move* sensors in Isaac, set USD prim paths in your YAML (e.g. `configs/default.yaml` or `configs/obstacle_isaaclab.yaml`) under:
- `sensor_models.lidar.isaac.prim_path`
- `sensor_models.camera.isaac.prim_path`

For parallel env cloning, paths often look like:
`/World/envs/env_{env_idx}/Robot/.../YourLidar`
`/World/envs/env_{env_idx}/Robot/.../YourDepthCamera`

The bridge will `.format(env_idx=...)` if `{env_idx}` is present.

## Metrics and configs

- **`--bridge-mode ground`** — `blind_spot_fraction` from **depth** + **3D point-like** arrays in observations (task-dependent). Falls back to `fast_baseline_metrics(...)` for that episode if perception is missing (one-time warning).
- **`--bridge-mode obstacle`** — research-style corridor metrics: `t_det_s_p95`, `collision_rate`, `safety_success` / `mean_goal_success` (no-contact + clearance). Pair with **`configs/obstacle_isaaclab.yaml`** and `inner_loop.n_episodes` / `max_steps_per_episode` consistent with the bridge’s `--max-steps` and `--sim-dt`.


In [ ]:
# Install Isaac Sim 4.5 and Isaac Lab 2.1.0. This process takes about 10 mins to complete.
!wget -O install-isaac-sim-4.5.sh https://raw.githubusercontent.com/j3soon/isaac-sim-colab/main/scripts/install-isaac-sim-4.5.sh
!time bash install-isaac-sim-4.5.sh
!wget -O install-isaac-lab-2.1.0.sh https://raw.githubusercontent.com/j3soon/isaac-sim-colab/main/scripts/install-isaac-lab-2.1.0.sh
!time bash install-isaac-lab-2.1.0.sh


In [ ]:
# Clone *this* repository (the sensor placement optimizer) and install its Python deps.
# NOTE: if you are iterating on a fork, change REPO_URL.

import os
import subprocess
import sys

REPO_URL = "https://github.com/dcaglar-28/sensor_placement_opt.git"
REPO_DIR = "/content/sensor_placement_opt"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # IMPORTANT: in Colab, use `{REPO_DIR}` (Python formatting), not `"$REPO_DIR"` in `!` shells.
    !rm -rf "{REPO_DIR}"
    !git clone "{REPO_URL}" "{REPO_DIR}"

os.chdir(REPO_DIR)

# Install into the *same interpreter as this kernel* (avoids "pip installed but import fails")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

print("OK:", os.getcwd())
print("python:", sys.executable)


In [ ]:
# --- Kernel dependency sanity check (run right after `pip install -r requirements.txt`) ---

import importlib
import sys

for mod in ("cma", "yaml", "numpy"):
    importlib.import_module(mod)

import cma

print("OK: imports work")
print("python:", sys.executable)
print("cma:", cma.__version__)


## Start the Isaac Lab HTTP bridge (background process)

`scripts/isaaclab_sensor_bridge.py` follows the Isaac Lab pattern: `AppLauncher` → `parse_env_cfg` → `gym.make` → `env.step`.

The next cell starts the bridge with **`subprocess` + this kernel’s `sys.executable`** (reliable in Colab). Set **`BRIDGE_MODE`** to `ground` or `obstacle` (or `os.environ["BRIDGE_MODE"]`). For **`obstacle`**, optional `D_WARN`, `D_CLEAR`, `SIM_DT`, and `MAX_STEPS` are passed through.

`PORT` and `NUM_ENVS` must match the optimizer cell below.

If you get import errors, re-run the Isaac install cell and use the same Python the installer expects.


In [ ]:
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request

# Same directory as the clone cell above
REPO_DIR = "/content/sensor_placement_opt"
os.chdir(REPO_DIR)

# Isaac Sim / Kit environment flags (per Isaac Sim install docs)
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

PORT = 8010
TASK = os.environ.get("ISAAC_TASK", "Isaac-Velocity-Flat-Unitree-Go2-v0")
NUM_ENVS = 1
BRIDGE_MODE = os.environ.get("BRIDGE_MODE", "ground")  # "ground" | "obstacle" | "generic"
MAX_STEPS = 500
# Obstacle-mode extras (ignored when BRIDGE_MODE != "obstacle")
D_WARN = 3.0
D_CLEAR = 0.5
SIM_DT = 0.02
OBSTACLE_ROOT = "/World/bridge_corridor"

BRIDGE = os.path.join(REPO_DIR, "scripts", "isaaclab_sensor_bridge.py")
LOG = "/tmp/isaaclab_sensor_bridge.log"

!pkill -f "isaaclab_sensor_bridge.py" 2>/dev/null || true

cmd = [
    sys.executable,
    BRIDGE,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--bridge-mode",
    BRIDGE_MODE,
    "--headless",
    "--enable_cameras",
    "--task",
    TASK,
    "--num_envs",
    str(NUM_ENVS),
    "--repo-root",
    REPO_DIR,
    "--max-steps",
    str(MAX_STEPS),
]
if BRIDGE_MODE == "obstacle":
    cmd += [
        "--d-warn",
        str(D_WARN),
        "--d-clear",
        str(D_CLEAR),
        "--sim-dt",
        str(SIM_DT),
        "--obstacle-root",
        OBSTACLE_ROOT,
    ]

_log = open(LOG, "wb")
_proc = subprocess.Popen(
    cmd,
    stdout=_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
_log.close()
print("Launched bridge pid=", _proc.pid, "mode=", BRIDGE_MODE, "log=", LOG, flush=True)

# The bridge only binds HTTP after `AppLauncher` + `gym.make` init; that can take many minutes in Colab.
deadline = time.time() + 60 * 30
last_log_pos = 0


def _read_new_log_bytes() -> bytes:
    global last_log_pos
    try:
        with open(LOG, "rb") as f:
            f.seek(last_log_pos)
            chunk = f.read()
            last_log_pos = f.tell()
            return chunk
    except FileNotFoundError:
        return b""


print("Waiting for HTTP bridge: http://127.0.0.1:%d/health ..." % PORT, flush=True)
while time.time() < deadline:
    chunk = _read_new_log_bytes()
    if chunk:
        print(chunk.decode("utf-8", errors="replace"), end="")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as r:
            body = r.read()
        print("Bridge is ready:", body.decode("utf-8", errors="replace").strip(), flush=True)
        break
    except (urllib.error.URLError, TimeoutError, ConnectionError, OSError):
        time.sleep(5)
else:
    raise RuntimeError("Timed out waiting for bridge /health. See log: " + LOG)

!tail -n 60 "{LOG}" || true
print("Bridge log:", LOG)


## Run the sensor placement optimizer against the local bridge

The JSON response from `/run_rollouts` includes all `EvalMetrics` fields (see `sensor_opt/loss/loss.py`). The client below maps every key so **`t_det_s_p95`**, **`safety_success`**, etc. are not dropped when using `--bridge-mode obstacle`.

By default this cell loads **`configs/default.yaml`** and runs a **tiny** CMA smoke (1 generation, small population). Set **`CONFIG_PATH`** to **`configs/obstacle_isaaclab.yaml`** when `BRIDGE_MODE=obstacle` so `loss.mode: obstacle_latency` matches the bridge metrics.


In [ ]:
import copy
import json
import os
import urllib.request
from dataclasses import asdict

import numpy as np
import yaml

from sensor_opt.encoding.config import SensorConfig
from sensor_opt.inner_loop.isaac_evaluator import IsaacSimEvaluator
from sensor_opt.logging.experiment_logger import ExperimentLogger
from sensor_opt.loss.loss import EvalMetrics
from sensor_opt.search.factory import create_search


def eval_metrics_from_bridge_row(m: dict) -> EvalMetrics:
    """Map full /run_rollouts JSON dict to EvalMetrics (all optional fields preserved)."""
    return EvalMetrics(
        collision_rate=float(m["collision_rate"]),
        blind_spot_fraction=float(m.get("blind_spot_fraction", 0.0)),
        mean_goal_success=float(m.get("mean_goal_success", 0.0)),
        n_episodes=int(m["n_episodes"]),
        t_det_s=float(m.get("t_det_s", 0.0)),
        t_det_s_p95=float(m.get("t_det_s_p95", 0.0)),
        episode_time_s=float(m.get("episode_time_s", 0.0)),
        safety_success=float(m.get("safety_success", 0.0)),
    )


class IsaacRpcEnvClient:
    """JSON-over-HTTP client matching `IsaacSimEvaluator`'s env contract."""

    def __init__(self, base_url: str, timeout_sec: float = 600.0):
        self.base_url = base_url.rstrip("/")
        self.timeout_sec = float(timeout_sec)

    def reconfigure_sensors(self, env_idx: int, config: SensorConfig, sensor_models: dict) -> None:
        payload = {"env_idx": int(env_idx), "config": asdict(config), "sensor_models": sensor_models}
        self._post_json("/reconfigure_sensors", payload)

    def run_rollouts(self, n_episodes: int, rng: np.random.Generator) -> list[EvalMetrics]:
        seed = int(rng.integers(0, 2**31 - 1))
        payload = {"n_episodes": int(n_episodes), "seed": int(seed)}
        data = self._post_json("/run_rollouts", payload)
        return [eval_metrics_from_bridge_row(m) for m in data["metrics"]]

    def _post_json(self, path: str, payload: dict):
        url = f"{self.base_url}{path}"
        req = urllib.request.Request(
            url,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=self.timeout_sec) as resp:
            return json.loads(resp.read().decode("utf-8"))


os.chdir("/content/sensor_placement_opt")

PORT = 8010  # must match the bridge cell
NUM_ENVS = 1  # must match --num_envs in the bridge cell
# Use `configs/obstacle_isaaclab.yaml` when the bridge runs with --bridge-mode obstacle
CONFIG_PATH = os.environ.get("CONFIG_PATH", "configs/default.yaml")

env = IsaacRpcEnvClient(base_url=f"http://127.0.0.1:{PORT}", timeout_sec=60 * 30)
base_evaluator = IsaacSimEvaluator(isaac_sim_cfg={"env": env, "num_envs": NUM_ENVS})

cfg = yaml.safe_load(open(CONFIG_PATH))
cfg = copy.deepcopy(cfg)
cfg["inner_loop"]["mode"] = "isaac_sim"
cfg["multi_fidelity"]["enabled"] = False

# tiny smoke settings
cfg["cma"]["max_generations"] = 1
cfg["cma"]["population_size"] = 4
cfg["cma"]["tolx"] = 1e-9
cfg["cma"]["tolfun"] = 1e-9

seed = int(cfg.get("experiment", {}).get("seed", 42))

with ExperimentLogger(
    experiment_name=cfg["experiment"]["name"] + "_isaaclab_smoke",
    results_dir=cfg.get("logging", {}).get("results_dir", "results"),
    use_mlflow=False,
    mlflow_tracking_uri=cfg.get("logging", {}).get("mlflow_tracking_uri", "mlruns"),
    run_config=cfg,
) as logger:
    search = create_search(
        search_type=cfg["search"]["type"],
        config=cfg,
        evaluator={
            "evaluator_fn": None,
            "evaluator": None,
            "base_evaluator": base_evaluator,
            "logger": logger,
            "seed": seed,
        },
    )
    result = search.run()

print("config:", CONFIG_PATH)
print("best_loss:", result.best_loss)
print("run_id:", result.run_id)
